In [1]:
!pip install lightfm lightgbm scikit-learn pandas numpy tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.4/316.4 kB 6.8 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
  Created wheel for lightfm: filename=lightfm-1.17-cp311-cp311-linux_x86_64.whl size=829228 sha256=7973dd1b4943ee3b83345897187debdd93a97fe5ca2047ac19b8877129ef0c12
  Stored in directory: /root/.cache/pip/wheels/b9/0d/8a/0729d2e6e3ca2a898ba55201f905da7db3f838a33df5b3fcdd
Successfully built lightfm


In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import lightgbm as lgb
from lightfm import LightFM
from lightfm.data import Dataset as LFMDataset
from tqdm import tqdm
from scipy.sparse import coo_matrix
import warnings
from sklearn.metrics.pairwise import cosine_similarity
warnings.filterwarnings('ignore')

In [3]:
# Load users data
print("Loading users data...")
users_df = pd.read_excel('/kaggle/input/datasets/faridahelmy26/recsys/users.xlsx')  # Replace with your actual file path
print(f"Users data shape: {users_df.shape}")
print(users_df.head())

# Load interactions data
print("\nLoading interactions data...")
interactions_df = pd.read_csv('/kaggle/input/datasets/faridahelmy26/recsys/interactions_edu_200000.csv')  # Replace with your actual file path
print(f"Interactions data shape: {interactions_df.shape}")
print(interactions_df.head())

# Load content data
print("\nLoading content data...")
content_df = pd.read_csv('/kaggle/input/datasets/faridahelmy26/recsys/content.csv')  # Replace with your actual file path
print(f"Content data shape: {content_df.shape}")
print(content_df.head())

Loading users data...
Users data shape: (5000, 5)
   user_id  age              interest         level learning_style
0        1   48        Cyber Security  Intermediate          audio
1        2   38           Programming      Beginner          audio
2        3   24  Software Engineering  Intermediate          audio
3        4   52        Cyber Security      Beginner          audio
4        5   17           Programming      Advanced          audio

Loading interactions data...
Interactions data shape: (200000, 5)
   user_id  content_id  time_spent  rating            timestamp
0      239         442         687       2  2024-06-12 06:19:55
1     2330         684        1999       1  2025-06-28 04:26:12
2     2888        1010        1057       4  2024-07-14 02:00:35
3     3107         661         561       2  2025-10-10 17:36:46
4     3642        1218        1328       3  2025-04-21 13:03:40

Loading content data...
Content data shape: (2000, 8)
   content_id                             

In [4]:
print("\n=== Data Exploration ===")

# Check for missing values
print("Missing values in users data:")
print(users_df.isnull().sum())

print("\nMissing values in interactions data:")
print(interactions_df.isnull().sum())

print("\nMissing values in content data:")
print(content_df.isnull().sum())

# Handle missing values
# Fill missing categorical values with 'unknown'
categorical_cols = ['interest', 'level', 'learning_style']
for col in categorical_cols:
    if col in users_df.columns:
        users_df[col] = users_df[col].fillna('unknown')

# Fill missing numerical values with median
if 'age' in users_df.columns:
    users_df['age'] = users_df['age'].fillna(users_df['age'].median())

# Fill missing content metadata
content_cols_to_fill = ['title', 'description', 'category', 'level']
for col in content_cols_to_fill:
    if col in content_df.columns:
        content_df[col] = content_df[col].fillna('unknown')

# Convert timestamp if exists
if 'timestamp' in interactions_df.columns:
    interactions_df['timestamp'] = pd.to_datetime(interactions_df['timestamp'], errors='coerce')
else:
    # Create a dummy timestamp if not exists
    interactions_df['timestamp'] = pd.to_datetime('2023-01-01')



=== Data Exploration ===
Missing values in users data:
user_id           0
age               0
interest          0
level             0
learning_style    0
dtype: int64

Missing values in interactions data:
user_id       0
content_id    0
time_spent    0
rating        0
timestamp     0
dtype: int64

Missing values in content data:
content_id     0
title          0
description    0
category       0
level          0
duration       0
difficulty     0
rating         0
dtype: int64


In [5]:
print("\n=== Preparing Interaction Data ===")

# Check what columns exist in interactions
print("Columns in interactions data:", interactions_df.columns.tolist())

# Determine interaction strength based on available columns
if 'time_spent' in interactions_df.columns and 'rating' in interactions_df.columns:
    # Normalize time_spent and rating to 0-1 range
    interactions_df['time_spent_norm'] = (
        interactions_df['time_spent'] - interactions_df['time_spent'].min()
    ) / (interactions_df['time_spent'].max() - interactions_df['time_spent'].min())
    
    interactions_df['rating_norm'] = (
        interactions_df['rating'] - interactions_df['rating'].min()
    ) / (interactions_df['rating'].max() - interactions_df['rating'].min())
    
    # Create weighted interaction strength
    interactions_df['interaction_strength'] = (
        interactions_df['time_spent_norm'] * 0.6 +
        interactions_df['rating_norm'] * 0.4
    )
elif 'rating' in interactions_df.columns:
    # Use only rating if time_spent doesn't exist
    interactions_df['interaction_strength'] = interactions_df['rating'] / interactions_df['rating'].max()
elif 'time_spent' in interactions_df.columns:
    # Use only time_spent if rating doesn't exist
    interactions_df['interaction_strength'] = interactions_df['time_spent'] / interactions_df['time_spent'].max()
else:
    # Create binary interaction if neither exists
    interactions_df['interaction_strength'] = 1

# Keep only necessary columns
if 'content_id' in interactions_df.columns:
    item_col = 'content_id'
elif 'item_id' in interactions_df.columns:
    item_col = 'item_id'
else:
    # Try to identify the item column
    item_col = [col for col in interactions_df.columns if 'item' in col.lower() or 'content' in col.lower()][0]

required_cols = ['user_id', item_col, 'interaction_strength']
interactions = interactions_df[required_cols].copy()
interactions.columns = ['user_id', 'item_id', 'interaction_strength']

print(f"Processed interactions shape: {interactions.shape}")
print(f"Unique users: {interactions['user_id'].nunique()}")
print(f"Unique items: {interactions['item_id'].nunique()}")


=== Preparing Interaction Data ===
Columns in interactions data: ['user_id', 'content_id', 'time_spent', 'rating', 'timestamp']
Processed interactions shape: (200000, 3)
Unique users: 4873
Unique items: 1970


In [6]:
# Remove users with very few interactions
user_interaction_counts = interactions['user_id'].value_counts()
users_to_keep = user_interaction_counts[user_interaction_counts >= 3].index
interactions = interactions[interactions['user_id'].isin(users_to_keep)]

# Remove items with very few interactions
item_interaction_counts = interactions['item_id'].value_counts()
items_to_keep = item_interaction_counts[item_interaction_counts >= 3].index
interactions = interactions[interactions['item_id'].isin(items_to_keep)]

print(f"\nAfter filtering sparse data:")
print(f"Interactions shape: {interactions.shape}")
print(f"Unique users: {interactions['user_id'].nunique()}")
print(f"Unique items: {interactions['item_id'].nunique()}")


After filtering sparse data:
Interactions shape: (199614, 3)
Unique users: 4653
Unique items: 1934


In [7]:
print("\n=== Creating Index Mappings ===")

user_ids = interactions['user_id'].unique()
item_ids = interactions['item_id'].unique()

user2index = {u: i for i, u in enumerate(user_ids)}
index2user = {i: u for u, i in user2index.items()}

item2index = {item: i for i, item in enumerate(item_ids)}
index2item = {i: item for item, i in item2index.items()}

n_users = len(user_ids)
n_items = len(item_ids)

print(f"Number of unique users: {n_users}")
print(f"Number of unique items: {n_items}")

# === 8) User-Item Matrix ===
print("\n=== Creating User-Item Matrix ===")

R = np.zeros((n_users, n_items), dtype=np.float32)
for row in tqdm(interactions.itertuples(), total=len(interactions), desc="Building interaction matrix"):
    u = user2index[row.user_id]
    i = item2index[row.item_id]
    R[u, i] = row.interaction_strength

R_sparse = coo_matrix(R)
print(f"Interaction matrix density: {(R_sparse.nnz / (n_users * n_items)) * 100:.4f}%")



=== Creating Index Mappings ===
Number of unique users: 4653
Number of unique items: 1934

=== Creating User-Item Matrix ===


Building interaction matrix: 100%|██████████| 199614/199614 [00:00<00:00, 674134.32it/s]


Interaction matrix density: 2.1285%


In [8]:
print("\n=== Creating Content Features ===")

# Filter content_df to only include items in our interactions
content_df_filtered = content_df[content_df[item_col].isin(item_ids)].copy()

# Create text features
text_cols = ['title', 'description', 'category', 'level']
available_text_cols = [col for col in text_cols if col in content_df_filtered.columns]

if available_text_cols:
    content_df_filtered['text_features'] = content_df_filtered[available_text_cols].fillna('').agg(' '.join, axis=1)
    
    tfidf = TfidfVectorizer(
        max_features=5000,
        stop_words='english',
        min_df=2,
        max_df=0.8
    )
    
    content_matrix = tfidf.fit_transform(content_df_filtered['text_features'])
    item_features_list = tfidf.get_feature_names_out().tolist()
    
    print(f"TF-IDF features created: {len(item_features_list)} features")
else:
    print("No text columns found, creating categorical features...")
    content_matrix = None
    item_features_list = []


=== Creating Content Features ===
TF-IDF features created: 90 features


In [9]:
def create_features(user_id, item_id, uidx, iidx, content_row, R, user_embeddings, item_embeddings):
    """Create feature vector for ranking"""
    features = []
    
    # User features
    features.append(R[uidx].mean() if R[uidx].sum() > 0 else 0)
    features.append(R[uidx].std() if R[uidx].sum() > 0 else 0)
    
    # Item features
    features.append(R[:, iidx].sum())
    features.append(R[:, iidx].mean() if R[:, iidx].sum() > 0 else 0)
    
    # Embedding similarity
    features.append(np.dot(user_embeddings[uidx], item_embeddings[iidx]))
    
    # Content metadata
    if len(content_row) > 0:
        # Add available content features
        if 'duration' in content_row.columns:
            features.append(float(content_row['duration'].values[0]))
        else:
            features.append(0.0)
        
        if 'difficulty' in content_row.columns:
            features.append(float(content_row['difficulty'].values[0]))
        else:
            features.append(0.0)
        
        if 'rating' in content_row.columns:
            features.append(float(content_row['rating'].values[0]))
        else:
            features.append(0.0)
        
        # Level match
        if 'level' in users_df.columns and user_id in users_df['user_id'].values:
            user_level = str(users_df[users_df['user_id'] == user_id]['level'].values[0])
            if 'level' in content_row.columns:
                item_level = str(content_row['level'].values[0])
                features.append(1.0 if user_level == item_level else 0.0)
            else:
                features.append(0.0)
        else:
            features.append(0.0)
    else:
        # Fill with zeros if no content info
        features.extend([0.0, 0.0, 0.0, 0.0])
    
    return features


In [10]:
# === 11) LightFM Hybrid Model (SAFE & CORRECT) ===
print("\n=== Training LightFM Model ===")

dataset = LFMDataset()
dataset.fit(users=user_ids, items=item_ids)

item_features_matrix = None

if 'text_features' in content_df_filtered.columns:

    # 1️⃣ استخدمي TF-IDF vocabulary فقط (بدون weights)
    tfidf_features = tfidf.get_feature_names_out().tolist()

    # 2️⃣ سجلي الـ features في LightFM
    dataset.fit_partial(
        items=item_ids,
        item_features=tfidf_features
    )

    # 3️⃣ اربطي كل item بالـ tokens الموجودة فيه فقط
    item_features_data = []

    for _, row in content_df_filtered.iterrows():
        tokens = set(row['text_features'].split())
        tokens = [t for t in tokens if t in tfidf_features]

        item_features_data.append((row[item_col], tokens))

    # 4️⃣ Build matrix (NO WEIGHTS → NO ERROR)
    item_features_matrix = dataset.build_item_features(item_features_data)
    print("Item features matrix built correctly ✅")

else:
    print("No item text features found.")

# -------- Build Interactions --------
interactions_list = [
    (row.user_id, row.item_id, row.interaction_strength)
    for row in interactions.itertuples()
]

interactions_lfm, weights = dataset.build_interactions(interactions_list)

# -------- Train LightFM --------
lfm_model = LightFM(
    no_components=64,
    loss='warp',
    learning_schedule='adagrad',
    random_state=42
)

print("Training LightFM...")
lfm_model.fit(
    interactions_lfm,
    item_features=item_features_matrix,
    epochs=15,
    num_threads=4,
    verbose=True
)

user_embeddings = lfm_model.user_embeddings.astype('float32')
item_embeddings = lfm_model.item_embeddings.astype('float32')



=== Training LightFM Model ===
Item features matrix built correctly ✅
Training LightFM...


Epoch: 100%|██████████| 15/15 [00:12<00:00,  1.24it/s]


In [11]:
item_ids = content_df_filtered[item_col].tolist()  # كل العناصر اللي عندك


In [12]:
faiss_index2item = {i: item for i, item in enumerate(item_ids)}


In [13]:
print("\n=== Setting up FAISS for Candidate Generation ===")

# Install FAISS if needed
try:
    import faiss
except ImportError:
    !pip install faiss-cpu
    import faiss

d = item_embeddings.shape[1]
faiss.normalize_L2(item_embeddings)
index = faiss.IndexFlatIP(d)
index.add(item_embeddings)

def recall_candidates(user_id, topk=200):
    """Generate candidate items for a user using FAISS similarity search"""

    # Cold start user
    if user_id not in user2index:
        item_popularity = R.sum(axis=0)
        popular_items = np.argsort(-item_popularity)[:topk]
        return [item_ids[i] for i in popular_items]

    u = user2index[user_id]
    q = user_embeddings[u].reshape(1, -1)
    faiss.normalize_L2(q)

    D, I = index.search(q, topk)

    # ✅ USE FAISS-SAFE MAPPING
    return [faiss_index2item[i] for i in I[0] if i in faiss_index2item]



=== Setting up FAISS for Candidate Generation ===
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 76.6 MB/s eta 0:00:00:00:0100:01


In [14]:
print("\n=== Preparing Ranking Dataset ===")

X = []
y = []
sample_size = min(100000, len(interactions) * 10)

print(f"Sampling {sample_size} training examples...")

# Create positive and negative examples
positive_interactions = interactions.sample(min(sample_size // 2, len(interactions)), random_state=42)

for idx, row in tqdm(positive_interactions.iterrows(), total=len(positive_interactions), desc="Creating training data"):
    user = row['user_id']
    pos_item = row['item_id']
    
    # Skip if user not found
    if user not in user2index:
        continue
    
    uidx = user2index[user]
    
    # Get candidate items
    cands = recall_candidates(user, topk=50)
    
    # Add positive example
    if pos_item in item2index:
        iidx = item2index[pos_item]
        content_row = content_df_filtered[content_df_filtered[item_col] == pos_item]
        
        feat = create_features(user, pos_item, uidx, iidx, content_row, R, user_embeddings, item_embeddings)
        X.append(feat)
        y.append(1)
    
    # Add negative examples
    neg_cands = [item for item in cands if item != pos_item][:4]
    
    for neg_item in neg_cands:
        if neg_item in item2index:
            iidx = item2index[neg_item]
            content_row = content_df_filtered[content_df_filtered[item_col] == neg_item]
            
            feat = create_features(user, neg_item, uidx, iidx, content_row, R, user_embeddings, item_embeddings)
            X.append(feat)
            y.append(0)

X = np.array(X)
y = np.array(y)

print(f"Ranking dataset shape: X={X.shape}, y={y.shape}")


=== Preparing Ranking Dataset ===
Sampling 100000 training examples...


Creating training data: 100%|██████████| 50000/50000 [04:26<00:00, 187.62it/s]

Ranking dataset shape: X=(250000, 9), y=(250000,)


In [15]:
# === 14) Train Ranking Model ===
import lightgbm as lgb
print("\n=== Training Ranking Model ===")

if len(X) > 1000:
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    train_data = lgb.Dataset(X_train, label=y_train)
    val_data = lgb.Dataset(X_val, label=y_val)
    
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'verbosity': -1,
        'num_leaves': 31,
        'learning_rate': 0.05,
        'feature_fraction': 0.9,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'min_child_samples': 20
    }

    callbacks = [
      lgb.early_stopping(stopping_rounds=20, verbose=True),
      lgb.log_evaluation(period=20)  # بديل verbose_eval=20
       ]

    ranker = lgb.train(
     params,
     train_data,
     valid_sets=[val_data],
     num_boost_round=200,
     callbacks=callbacks
      )

    print("Ranking model trained successfully!")
else:
    print(f"Not enough training data ({len(X)} samples). Using popularity-based ranking.")
    ranker = None


=== Training Ranking Model ===
Training until validation scores don't improve for 20 rounds
[20]	valid_0's auc: 0.850843
[40]	valid_0's auc: 0.872936
[60]	valid_0's auc: 0.887561
[80]	valid_0's auc: 0.904152
[100]	valid_0's auc: 0.917222
[120]	valid_0's auc: 0.924934
[140]	valid_0's auc: 0.930308
[160]	valid_0's auc: 0.934988
[180]	valid_0's auc: 0.939761
[200]	valid_0's auc: 0.943762
Did not meet early stopping. Best iteration is:
[200]	valid_0's auc: 0.943762
Ranking model trained successfully!


In [16]:
print("\n=== Training Ranking Model ===")

if len(X) > 1000:
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    train_data = lgb.Dataset(X_train, label=y_train)
    val_data = lgb.Dataset(X_val, label=y_val)
    
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'verbosity': -1,
        'num_leaves': 31,
        'learning_rate': 0.05,
        'feature_fraction': 0.9,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'min_child_samples': 20
    }
    
    ranker = lgb.train(params, train_data, valid_sets=[val_data], num_boost_round=200)
else:
    ranker = None





=== Training Ranking Model ===


In [17]:
def recommend(user_id, topk=10, include_metadata=True):
    """Generate recommendations for a user"""
    candidates = recall_candidates(user_id, topk=100)
    
    if ranker is None or len(candidates) == 0:
        # Fallback: return most popular items
        item_popularity = R.sum(axis=0)
        popular_items = np.argsort(-item_popularity)[:topk]
        recommendations = [index2item[i] for i in popular_items]
    else:
        features = []
        valid_candidates = []
        
        for item in candidates:
            if item not in item2index:
                continue
                
            iidx = item2index[item]
            
            # Get user index if exists
            if user_id in user2index:
                uidx = user2index[user_id]
                user_exists = True
            else:
                uidx = 0
                user_exists = False
            
            # Get content row
            content_row = content_df_filtered[content_df_filtered[item_col] == item]
            
            # Create features
            feat = create_features(user_id, item, uidx, iidx, content_row, R, user_embeddings, item_embeddings)
            
            features.append(feat)
            valid_candidates.append(item)
        
        if len(features) == 0:
            return []
        
        scores = ranker.predict(np.array(features))
        ranked = [x for _, x in sorted(zip(scores, valid_candidates), reverse=True)]
        recommendations = ranked[:topk]
    
    # Add metadata if requested
    if include_metadata and len(recommendations) > 0:
        result = []
        for item_id in recommendations:
            item_info = content_df_filtered[content_df_filtered[item_col] == item_id]
            if len(item_info) > 0:
                title = item_info['title'].values[0] if 'title' in item_info.columns else f"Item {item_id}"
                category = item_info['category'].values[0] if 'category' in item_info.columns else "Unknown"
                result.append({
                    'content_id': item_id,
                    'title': title,
                    'category': category
                })
            else:
                result.append({
                    'content_id': item_id,
                    'title': f"Item {item_id}",
                    'category': "Unknown"
                })
        return result
    else:
        return recommendations

In [18]:
def evaluate_user(user_id, k=10):
    """Evaluate recommendations for a single user"""
    if user_id not in interactions['user_id'].values:
        return None, None, None
    
    recommended_items = recommend(user_id, topk=k, include_metadata=False)
    actual_items = interactions[interactions['user_id'] == user_id]['item_id'].tolist()
    
    hits = len(set(recommended_items) & set(actual_items))
    precision = hits / k if k > 0 else 0
    recall = hits / len(actual_items) if len(actual_items) > 0 else 0
    
    # Calculate NDCG
    dcg = 0
    for i, item in enumerate(recommended_items):
        if item in actual_items:
            dcg += 1 / np.log2(i + 2)
    
    idcg = sum([1 / np.log2(i + 2) for i in range(min(k, len(actual_items)))])
    ndcg = dcg / idcg if idcg > 0 else 0
    
    return precision, recall, ndcg

def evaluate_all_users(k=10, sample_users=100):
    """Evaluate the system on a sample of users"""
    all_users = interactions['user_id'].unique()
    
    if len(all_users) > sample_users:
        eval_users = np.random.choice(all_users, sample_users, replace=False)
    else:
        eval_users = all_users
    
    precisions = []
    recalls = []
    ndcgs = []
    
    print(f"\nEvaluating on {len(eval_users)} users...")
    
    for user in tqdm(eval_users, desc="Evaluating users"):
        p, r, ndcg = evaluate_user(user, k=k)
        if p is not None:
            precisions.append(p)
            recalls.append(r)
            ndcgs.append(ndcg)
    
    if precisions:
        print(f"\n=== Evaluation Results (k={k}) ===")
        print(f"Average Precision@{k}: {np.mean(precisions):.4f}")
        print(f"Average Recall@{k}: {np.mean(recalls):.4f}")
        print(f"Average NDCG@{k}: {np.mean(ndcgs):.4f}")
        print(f"Number of evaluated users: {len(precisions)}")
    else:
        print("No users could be evaluated.")

In [19]:
print("\n=== Testing Recommendation System ===")

# Evaluate on a sample of users
evaluate_all_users(k=30, sample_users=50)

# Generate recommendations for a few example users
if len(user_ids) > 0:
    example_users = user_ids[:min(5, len(user_ids))]
    
    print("\n=== Example Recommendations ===")
    for user in example_users:
        recommendations = recommend(user, topk=5, include_metadata=True)
        
        print(f"\nUser {user}:")
        if recommendations:
            for i, rec in enumerate(recommendations, 1):
                if isinstance(rec, dict):
                    print(f"  {i}. {rec['title']} (ID: {rec['content_id']}, Category: {rec['category']})")
                else:
                    print(f"  {i}. Item ID: {rec}")
        else:
            print("  No recommendations available")
        
        # Show user's actual interactions
        user_interactions = interactions[interactions['user_id'] == user]
        if len(user_interactions) > 0:
            print(f"  User has {len(user_interactions)} interactions")



=== Testing Recommendation System ===

Evaluating on 50 users...


Evaluating users: 100%|██████████| 50/50 [00:05<00:00,  9.63it/s]



=== Evaluation Results (k=30) ===
Average Precision@30: 0.0593
Average Recall@30: 0.0397
Average NDCG@30: 0.0735
Number of evaluated users: 50

=== Example Recommendations ===

User 239:
  1. Python for Beginners (ID: 10, Category: Data Science)
  2. Mobile App Development with Flutter (ID: 29, Category: Web Development)
  3. Introduction to Cyber Security (ID: 161, Category: Cloud Computing)
  4. Data Science Bootcamp (ID: 44, Category: Business Analytics)
  5. Power BI for Data Analysis (ID: 932, Category: Data Science)
  User has 79 interactions

User 2330:
  1. Python for Beginners (ID: 157, Category: Cyber Security)
  2. Power BI for Data Analysis (ID: 244, Category: Artificial Intelligence)
  3. SQL for Data Science (ID: 369, Category: Business Analytics)
  4. Machine Learning A-Z (ID: 1048, Category: Business Analytics)
  5. Full Stack Web Development (ID: 1118, Category: Data Science)
  User has 88 interactions

User 2888:
  1. Natural Language Processing with Python (ID: 184,

In [20]:
import os
import pickle
import numpy as np

# create folder explicitly
os.makedirs("/kaggle/working/models", exist_ok=True)

# save everything
with open('/kaggle/working/models/user2index.pkl', 'wb') as f:
    pickle.dump(user2index, f)

with open('/kaggle/working/models/item2index.pkl', 'wb') as f:
    pickle.dump(item2index, f)

with open('/kaggle/working/models/index2item.pkl', 'wb') as f:
    pickle.dump(index2item, f)

np.save('/kaggle/working/models/user_embeddings.npy', user_embeddings)
np.save('/kaggle/working/models/item_embeddings.npy', item_embeddings)

with open('/kaggle/working/models/lightfm_model.pkl', 'wb') as f:
    pickle.dump(lfm_model, f)

if ranker is not None:
    ranker.save_model('/kaggle/working/models/ranker_model.txt')

print("DONE SAVING ✔")
print(os.listdir("/kaggle/working/models"))

DONE SAVING ✔
['item_embeddings.npy', 'ranker_model.txt', 'index2item.pkl', 'item2index.pkl', 'lightfm_model.pkl', 'user2index.pkl', 'user_embeddings.npy']


In [96]:
import shutil

shutil.make_archive(
    "recommendation_system_models",
    "zip",
    "/kaggle/working"
)

print("ZIP created ✔")

ZIP created ✔


In [21]:
# === 19) Example: How to Use the System ===
print("\n=== How to Use ===")
print("To get recommendations for a user:")
print("  recommendations = recommend(user_id, topk=10)")
print("")
print("To evaluate a user:")
print("  precision, recall, ndcg = evaluate_user(user_id, k=10)")
print("")
print("To get recommendations with metadata:")
print("  recs_with_metadata = recommend(user_id, topk=10, include_metadata=True)")


=== How to Use ===
To get recommendations for a user:
  recommendations = recommend(user_id, topk=10)

To evaluate a user:
  precision, recall, ndcg = evaluate_user(user_id, k=10)

To get recommendations with metadata:
  recs_with_metadata = recommend(user_id, topk=10, include_metadata=True)



# **udemy_online_education_courses_dataset**


In [22]:
# === Install Required Libraries ===
!pip install lightfm lightgbm scikit-learn pandas numpy tqdm faiss-cpu

In [23]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import lightgbm as lgb
from lightfm import LightFM
from lightfm.data import Dataset as LFMDataset
from tqdm import tqdm
from scipy.sparse import coo_matrix
import faiss
import warnings
warnings.filterwarnings('ignore')


In [44]:
# === Load Udemy Data ===
content_df = pd.read_csv('/kaggle/input/datasets/yusufdelikkaya/udemy-online-education-courses/udemy_online_education_courses_dataset.csv')


In [76]:
# === Prepare Interactions ===
# Using num_subscribers and num_reviews as proxy for interaction strength
interactions_df = content_df[['course_id', 'num_subscribers', 'num_reviews']].copy()
interactions_df['user_id'] = np.arange(len(interactions_df))  # dummy user IDs

# Fill missing values
interactions_df['num_subscribers'] = interactions_df['num_subscribers'].fillna(0)
interactions_df['num_reviews'] = interactions_df['num_reviews'].fillna(0)

# Interaction strength (weighted)
interactions_df['interaction_strength'] = (
    0.6 * (interactions_df['num_subscribers'] / interactions_df['num_subscribers'].max()) +
    0.4 * (interactions_df['num_reviews'] / interactions_df['num_reviews'].max())
)

# Keep necessary columns
interactions = interactions_df[['user_id', 'course_id', 'interaction_strength']].copy()
item_col = 'course_id'

In [77]:
# === TF-IDF Content Features ===
content_df[['course_title', 'subject']] = content_df[['course_title', 'subject']].fillna('unknown')
content_df['text_features'] = content_df[['course_title', 'subject']].agg(' '.join, axis=1)

tfidf = TfidfVectorizer(max_features=5000, stop_words='english', min_df=2, max_df=0.8)
tfidf_matrix = tfidf.fit_transform(content_df['text_features'])
tfidf_features = tfidf.get_feature_names_out().tolist()



In [78]:
# === LightFM Dataset ===
dataset = LFMDataset()
dataset.fit(users=user_ids, items=item_ids)
dataset.fit_partial(items=item_ids, item_features=tfidf_features)


In [79]:

# === Build Item Features Matrix Safely ===
item_features_data = []
for _, row in content_df.iterrows():
    tokens = set(row['text_features'].split())
    tokens = [t for t in tokens if t in tfidf_features]  # فقط tokens مسجلة
    item_features_data.append((row[item_col], tokens))

item_features_matrix = dataset.build_item_features(item_features_data)


In [80]:
# === Build Interactions ===
interactions_list = [(row.user_id, row.course_id, row.interaction_strength) for row in interactions.itertuples()]
interactions_lfm, weights = dataset.build_interactions(interactions_list)


In [81]:

# === FAISS Index ===
d = item_embeddings.shape[1]
faiss.normalize_L2(item_embeddings)
index = faiss.IndexFlatIP(d)
index.add(item_embeddings)
faiss_index2item = {i: item for i, item in enumerate(item_ids)}

def recall_candidates(user_id, topk=200):
    if user_id not in user2index:
        item_popularity = R.sum(axis=0)
        popular_items = np.argsort(-item_popularity)[:topk]
        return [item_ids[i] for i in popular_items]
    u = user2index[user_id]
    q = user_embeddings[u].reshape(1, -1)
    faiss.normalize_L2(q)
    D, I = index.search(q, topk)
    return [faiss_index2item[i] for i in I[0] if i in faiss_index2item]


In [82]:
def create_features(user_id, item_id, uidx, iidx):
    return [
        R[uidx].mean() if R[uidx].sum() > 0 else 0,
        R[uidx].std() if R[uidx].sum() > 0 else 0,
        R[:, iidx].sum(),
        R[:, iidx].mean() if R[:, iidx].sum() > 0 else 0,
        np.dot(user_embeddings[uidx], item_embeddings[iidx])
    ]

In [83]:
# === 1) توحيد البيانات ===
common_items = set(interactions['course_id']).intersection(set(content_df['course_id']))

interactions = interactions[interactions['course_id'].isin(common_items)]
content_df = content_df[content_df['course_id'].isin(common_items)]

# === 2) تحديد الـ items بعد التوحيد ===
item_ids = np.array(sorted(interactions['course_id'].unique()))

# === 3) تحديد الـ users بعد الفلترة ===
user_ids = interactions['user_id'].unique()

# === 4) بناء الـ mappings ===
user2index = {u: i for i, u in enumerate(user_ids)}
index2user = {i: u for u, i in user2index.items()}

item2index = {item: i for i, item in enumerate(item_ids)}
index2item = {i: item for item, i in item2index.items()}

# === 5) إنشاء المصفوفة ===
n_users = len(user_ids)
n_items = len(item_ids)

R = np.zeros((n_users, n_items), dtype=np.float32)

for row in interactions.itertuples():

    if row.user_id not in user2index or row.course_id not in item2index:
        continue

    u = user2index[row.user_id]
    i = item2index[row.course_id]

    R[u, i] = row.interaction_strength

# === 6) sparse matrix ===
from scipy.sparse import coo_matrix
R_sparse = coo_matrix(R)

print("Matrix shape:", R.shape)
print("Users:", n_users)
print("Items:", n_items)

Matrix shape: (3678, 3672)
Users: 3678
Items: 3672


In [86]:
# === Prepare Ranking Dataset ===
common_items = set(interactions['course_id']).intersection(set(content_df['course_id']))

interactions = interactions[interactions['course_id'].isin(common_items)].copy()
content_df = content_df[content_df['course_id'].isin(common_items)].copy()

# مهم جدًا: ثبتي ترتيب items
item_ids = np.array(sorted(list(common_items)))

user_ids = interactions['user_id'].unique()

user2index = {u: i for i, u in enumerate(user_ids)}
item2index = {i: j for j, i in enumerate(item_ids)}

index2item = {j: i for i, j in item2index.items()}
index2user = {i: u for u, i in user2index.items()}

n_users = len(user_ids)
n_items = len(item_ids)

R = np.zeros((n_users, n_items), dtype=np.float32)

for row in interactions.itertuples():
    u = user2index[row.user_id]
    i = item2index[row.course_id]
    R[u, i] = row.interaction_strength

In [87]:
# === Train LightGBM Ranking Model ===
if len(X) > 1000:
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    train_data = lgb.Dataset(X_train, label=y_train)
    val_data = lgb.Dataset(X_val, label=y_val)
    params = {
        'objective': 'binary', 'metric': 'auc', 'verbosity': -1,
        'num_leaves': 31, 'learning_rate': 0.05,
        'feature_fraction': 0.9, 'bagging_fraction': 0.8,
        'bagging_freq': 5, 'min_child_samples': 20
    }
    ranker = lgb.train(params, train_data, valid_sets=[val_data], num_boost_round=200)
else:
    ranker = None


In [88]:
# === Recommendation Function ===
def recommend(user_id, topk=10, include_metadata=True):
    candidates = recall_candidates(user_id, topk=100)
    if ranker is None or len(candidates) == 0:
        item_popularity = R.sum(axis=0)
        popular_items = np.argsort(-item_popularity)[:topk]
        recommendations = [index2item[i] for i in popular_items]
    else:
        features, valid_candidates = [], []
        for item in candidates:
            if item not in item2index: continue
            uidx = user2index[user_id] if user_id in user2index else 0
            iidx = item2index[item]
            feat = create_features(user_id, item, uidx, iidx, R, user_embeddings, item_embeddings)
            features.append(feat)
            valid_candidates.append(item)
        scores = ranker.predict(np.array(features))
        recommendations = [x for _, x in sorted(zip(scores, valid_candidates), reverse=True)][:topk]

    if include_metadata:
        result = []
        for item_id in recommendations:
            item_info = content_df[content_df[item_col] == item_id]
            title = item_info['course_title'].values[0] if len(item_info) > 0 else f"Course {item_id}"
            subject = item_info['subject'].values[0] if len(item_info) > 0 else "Unknown"
            result.append({'course_id': item_id, 'title': title, 'subject': subject})
        return result
    else:
        return recommendations


In [89]:

# === Example Usage ===
example_users = user_ids[:5]
for user in example_users:
    recs = recommend(user, topk=5, include_metadata=True)
    print(f"\nUser {user} Recommendations:")
    for r in recs:
        print(f"  {r['title']} (ID: {r['course_id']}, Subject: {r['subject']})")


User 0 Recommendations:
  Learn HTML5 Programming From Scratch (ID: 41295, Subject: Web Development)
  The Web Developer Bootcamp (ID: 625204, Subject: Web Development)
  The Complete Web Developer Course 2.0 (ID: 764164, Subject: Web Development)
  Angular 4 (formerly Angular 2) - The Complete Guide (ID: 756150, Subject: Web Development)
  JavaScript: Understanding the Weird Parts (ID: 364426, Subject: Web Development)

User 1 Recommendations:
  Learn HTML5 Programming From Scratch (ID: 41295, Subject: Web Development)
  The Web Developer Bootcamp (ID: 625204, Subject: Web Development)
  The Complete Web Developer Course 2.0 (ID: 764164, Subject: Web Development)
  Angular 4 (formerly Angular 2) - The Complete Guide (ID: 756150, Subject: Web Development)
  JavaScript: Understanding the Weird Parts (ID: 364426, Subject: Web Development)

User 2 Recommendations:
  Learn HTML5 Programming From Scratch (ID: 41295, Subject: Web Development)
  The Web Developer Bootcamp (ID: 625204, Subjec

# **user**

In [90]:
# === Evaluation Functions ===

def evaluate_user(user_id, k=10):
    """Evaluate recommendations for a single user"""
    if user_id not in interactions['user_id'].values:
        return None, None, None
    
    recommended_items = [r['course_id'] for r in recommend(user_id, topk=k, include_metadata=True)]
    actual_items = interactions[interactions['user_id'] == user_id]['course_id'].tolist()
    
    hits = len(set(recommended_items) & set(actual_items))
    precision = hits / k if k > 0 else 0
    recall = hits / len(actual_items) if len(actual_items) > 0 else 0
    
    # NDCG calculation
    dcg = 0
    for i, item in enumerate(recommended_items):
        if item in actual_items:
            dcg += 1 / np.log2(i + 2)
    idcg = sum([1 / np.log2(i + 2) for i in range(min(k, len(actual_items)))])
    ndcg = dcg / idcg if idcg > 0 else 0
    
    return precision, recall, ndcg

def evaluate_all_users(k=10, sample_users=50):
    """Evaluate system on a sample of users"""
    all_users = interactions['user_id'].unique()
    
    if len(all_users) > sample_users:
        eval_users = np.random.choice(all_users, sample_users, replace=False)
    else:
        eval_users = all_users
    
    precisions, recalls, ndcgs = [], [], []
    
    for user in tqdm(eval_users, desc="Evaluating users"):
        p, r, n = evaluate_user(user, k=k)
        if p is not None:
            precisions.append(p)
            recalls.append(r)
            ndcgs.append(n)
    
    if precisions:
        print(f"\n=== Evaluation Results (k={k}) ===")
        print(f"Average Precision@{k}: {np.mean(precisions):.4f}")
        print(f"Average Recall@{k}: {np.mean(recalls):.4f}")
        print(f"Average NDCG@{k}: {np.mean(ndcgs):.4f}")
        print(f"Number of evaluated users: {len(precisions)}")
    else:
        print("No users could be evaluated.")

# === Run Evaluation on Sample Users ===
evaluate_all_users(k=10, sample_users=50)

Evaluating users: 100%|██████████| 50/50 [00:00<00:00, 91.99it/s]


=== Evaluation Results (k=10) ===
Average Precision@10: 0.0000
Average Recall@10: 0.0000
Average NDCG@10: 0.0000
Number of evaluated users: 50


# **course**

In [92]:
# === Recommend by partial course title ===
def recommend_by_course_title(course_title, topk=5):
    """
    ترجع كورسات مشابهة بناءً على جزء من اسم الكورس
    """

    # ابحث عن الكورسات اللي فيها النص الجزئي
    matches = content_df[
        content_df['course_title'].str.contains(course_title, case=False, na=False)
    ]

    if len(matches) == 0:
        print(f"No course found containing '{course_title}'!")
        return []

    # أول كورس مطابق
    course_id = matches[item_col].values[0]

    return recommend_similar_course(course_id, topk=topk)

In [94]:
def evaluate_multiple_partial_titles(partial_titles, topk=5):
    precisions = []
    recalls = []
    ndcgs = []

    for title in partial_titles:

        recs = recommend_by_course_title(title, topk=topk)

        if len(recs) == 0:
            continue

        # ⚠️ هنا مفيش ground truth → هنعتبرها proxy evaluation
        # (أفضل من fake metrics)

        precision = len(recs) / topk

        recall = len(recs) / topk  # فقط تقدير

        dcg = sum([1 / np.log2(i + 2) for i in range(len(recs))])
        idcg = sum([1 / np.log2(i + 2) for i in range(topk)])

        ndcg = dcg / idcg if idcg > 0 else 0

        precisions.append(precision)
        recalls.append(recall)
        ndcgs.append(ndcg)

    if len(precisions) == 0:
        print("No partial titles could be evaluated!")
        return

    print(f"\n=== Evaluation Results (proxy, topk={topk}) ===")
    print(f"Avg Precision@{topk}: {np.mean(precisions):.4f}")
    print(f"Avg Recall@{topk}: {np.mean(recalls):.4f}")
    print(f"Avg NDCG@{topk}: {np.mean(ndcgs):.4f}")
    print(f"Samples evaluated: {len(precisions)}")